# 0826_dongjin_027_global_ensemble_shap

This notebook performs Global SHAP analysis natively on the **saved** `0825_peace_005_type_expert_fold_ensemble.pkl` model bundle.
Instead of filtering only False Calls, we compute SHAP values over the **entire Test Set** for each inspection type.
This allows us to identify the most dominant features the model relies on to separate `class = 0` (False Calls/Negatives) from `class = 1` (True Defects).

For each inspection type:
1. We compute SHAP values for the entire test set.
2. We aggregate Mean |SHAP| across all 4 checkpoint models.
3. We extract decision boundaries for the globally dominant features.


In [1]:
import gc
import json
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import shap
import xgboost
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Configuration
DATA_PATH = Path("../data/raw/dataset.csv")
MAPPING_PATH = Path("../data/raw/mapping.json")
MODEL_BUNDLE_PATH = Path("../models/0825_peace_005_type_expert_fold_ensemble.pkl")
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"


E:\pro_newton\제조 AI\팀 과제\siemens_aoi_ML_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data, Mapping, and Model Bundle

In [2]:
# Load Model Bundle
with open(MODEL_BUNDLE_PATH, 'rb') as f:
    bundle = pickle.load(f)

inspection_types = bundle['inspection_types']
feature_columns_by_type = bundle['feature_columns_by_type']
checkpoints = bundle['ensemble_checkpoints']

# Load Data
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
if raw_df.columns[0].startswith("Unnamed:") or raw_df.columns[0] == "":
    raw_df = raw_df.rename(columns={raw_df.columns[0]: RECORD_ID})

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

meta_columns = [col for col in raw_df.columns if col.startswith("meta_feat")]

print("Model and Data loaded. Types:", inspection_types)


Model and Data loaded. Types: [0, 1, 2, 3, 4]


## 2. Test Set Split

In [3]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index

def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]

validation_end_time = boundary_at(0.80)
test_mask = raw_df[TIME_COLUMN] > validation_end_time
test_df = raw_df.loc[test_mask].copy()

print(f"Test Set Size: {len(test_df)}")


Test Set Size: 88052


## 3. Extract Global SHAP per Type

In [4]:
final_summary_rows = []

for inspection_type in inspection_types:
    print("=" * 80)
    print(f"INSPECTION TYPE: {inspection_type}")
    print("=" * 80)
    
    feature_columns = feature_columns_by_type[inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    y_test_type = type_test[TARGET].astype("int8")
    
    if len(type_test) == 0:
        continue
        
    X_type_raw = type_test[feature_columns]
    
    print(f"\n  Total Samples (Test Set): {len(X_type_raw)}")
    print(f"  Negatives (class=0): {(y_test_type == 0).sum()}")
    print(f"  Positives (class=1): {(y_test_type == 1).sum()}\n")
        
    # SHAP Analysis
    feature_abs_shap_sum = {}
    all_trees = []
    
    for ckpt in checkpoints:
        model_info = bundle['members'][ckpt][inspection_type]
        model = model_info['model']
        preprocessor = model_info['preprocessor']
        encoded_feature_names = preprocessor.get_feature_names_out()
        
        X_type_encoded = preprocessor.transform(X_type_raw)
        if hasattr(X_type_encoded, "toarray"):
            X_type_encoded = X_type_encoded.toarray()
            
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_type_encoded)
        
        # Mean absolute SHAP for this checkpoint across ALL test samples
        mean_abs_shap_ckpt = np.abs(shap_values).mean(axis=0)
        
        for name, val in zip(encoded_feature_names, mean_abs_shap_ckpt):
            feature_abs_shap_sum[name] = feature_abs_shap_sum.get(name, 0.0) + val
            
        # Thresholds
        trees_df = model.get_booster().trees_to_dataframe()
        feature_map = {f"f{i}": name for i, name in enumerate(encoded_feature_names)}
        trees_df['FeatureName'] = trees_df['Feature'].map(feature_map)
        all_trees.append(trees_df)
        
    # Average across the 4 checkpoints
    for name in feature_abs_shap_sum:
        feature_abs_shap_sum[name] /= len(checkpoints)
        
    # Sort and get top 10
    top_features = sorted(feature_abs_shap_sum.keys(), key=lambda k: feature_abs_shap_sum[k], reverse=True)[:10]
    
    print("  Global Top Features (according to Mean Ensemble |SHAP|):")
    top_summary = []
    for i, feature in enumerate(top_features, 1):
        shap_val = feature_abs_shap_sum[feature]
        top_summary.append(f"{feature} ({shap_val:.4f})")
        print(f"    {i}. {feature} (Mean |SHAP|: {shap_val:.4f})")
        
    final_summary_rows.append({
        "Inspection Type": inspection_type,
        "Test Samples": len(X_type_raw),
        "Global Top Features (|SHAP|)": " <br> ".join([f"{i}. {val}" for i, val in enumerate(top_summary, 1)])
    })
    
    # Thresholds
    print("\n  -- Native Thresholds for Top Features (Aggregated) --")
    merged_trees = pd.concat(all_trees, ignore_index=True)
    
    for feature in top_features:
        feature_nodes = merged_trees[merged_trees['FeatureName'] == feature]
        if len(feature_nodes) == 0:
            continue
            
        threshold_agg = feature_nodes.groupby('Split').agg(
            Frequency=('Split', 'count'),
            Total_Cover=('Cover', 'sum')
        ).sort_values(by='Total_Cover', ascending=False)
        
        top_5 = threshold_agg.head(3)
        thresholds_str = ", ".join([f"< {th:.4f} (Cover: {row['Total_Cover']:.0f})" for th, row in top_5.iterrows()])
        print(f"    * {feature}: {thresholds_str}")
        
    print("\n")


INSPECTION TYPE: 0

  Total Samples (Test Set): 19491
  Negatives (class=0): 19296
  Positives (class=1): 195



  Global Top Features (according to Mean Ensemble |SHAP|):
    1. continuous__inspection_feat41 (Mean |SHAP|: 0.8916)
    2. categorical__meta_feat2_1 (Mean |SHAP|: 0.7688)
    3. continuous__inspection_feat24 (Mean |SHAP|: 0.5144)
    4. categorical__meta_feat4_0 (Mean |SHAP|: 0.4432)
    5. categorical__meta_feat1_10 (Mean |SHAP|: 0.3970)
    6. continuous__inspection_feat25 (Mean |SHAP|: 0.2829)
    7. continuous__inspection_feat23 (Mean |SHAP|: 0.2733)
    8. categorical__meta_feat1_12 (Mean |SHAP|: 0.1898)
    9. continuous__inspection_feat26 (Mean |SHAP|: 0.1814)
    10. continuous__inspection_feat42 (Mean |SHAP|: 0.1582)

  -- Native Thresholds for Top Features (Aggregated) --
    * continuous__inspection_feat41: < 0.4833 (Cover: 3539), < 0.7167 (Cover: 3203), < 0.6333 (Cover: 1352)
    * categorical__meta_feat2_1: < 2.0000 (Cover: 14470)
    * continuous__inspection_feat24: < 0.4542 (Cover: 4976), < 0.5250 (Cover: 3131), < 0.1958 (Cover: 2345)
    * categorical__meta_feat4_0: <

  Global Top Features (according to Mean Ensemble |SHAP|):
    1. continuous__inspection_feat48 (Mean |SHAP|: 1.2720)
    2. continuous__inspection_feat24 (Mean |SHAP|: 0.6964)
    3. categorical__meta_feat1_22 (Mean |SHAP|: 0.5671)
    4. categorical__meta_feat4_28 (Mean |SHAP|: 0.4157)
    5. continuous__inspection_feat4 (Mean |SHAP|: 0.3447)
    6. continuous__inspection_feat3 (Mean |SHAP|: 0.3380)
    7. continuous__inspection_feat8 (Mean |SHAP|: 0.3031)
    8. continuous__inspection_feat1 (Mean |SHAP|: 0.2875)
    9. continuous__inspection_feat25 (Mean |SHAP|: 0.2837)
    10. continuous__inspection_feat5 (Mean |SHAP|: 0.2361)

  -- Native Thresholds for Top Features (Aggregated) --
    * continuous__inspection_feat48: < 0.0708 (Cover: 11634), < 0.1833 (Cover: 6707), < 0.1167 (Cover: 6693)
    * continuous__inspection_feat24: < 0.0333 (Cover: 13019), < 0.6125 (Cover: 5379), < 0.2167 (Cover: 4896)
    * categorical__meta_feat1_22: < 2.0000 (Cover: 18322)
    * categorical__meta_feat

  Global Top Features (according to Mean Ensemble |SHAP|):
    1. categorical__meta_feat1_27 (Mean |SHAP|: 1.0023)
    2. continuous__inspection_feat96 (Mean |SHAP|: 0.9118)
    3. continuous__inspection_feat95 (Mean |SHAP|: 0.6849)
    4. continuous__inspection_feat12 (Mean |SHAP|: 0.6188)
    5. continuous__inspection_feat22 (Mean |SHAP|: 0.5021)
    6. categorical__meta_feat4_41 (Mean |SHAP|: 0.4317)
    7. continuous__inspection_feat1 (Mean |SHAP|: 0.3846)
    8. categorical__meta_feat4_3 (Mean |SHAP|: 0.3536)
    9. continuous__inspection_feat18 (Mean |SHAP|: 0.3235)
    10. categorical__meta_feat1_19 (Mean |SHAP|: 0.3234)

  -- Native Thresholds for Top Features (Aggregated) --
    * categorical__meta_feat1_27: < 2.0000 (Cover: 44604)
    * continuous__inspection_feat96: < 0.2653 (Cover: 27467), < 0.3163 (Cover: 19293), < 0.3551 (Cover: 17733)
    * continuous__inspection_feat95: < 0.2692 (Cover: 30689), < 0.3846 (Cover: 24852), < 0.5385 (Cover: 21505)
    * continuous__inspectio

  Global Top Features (according to Mean Ensemble |SHAP|):
    1. categorical__meta_feat4_7 (Mean |SHAP|: 0.7811)
    2. continuous__inspection_feat95 (Mean |SHAP|: 0.7195)
    3. continuous__inspection_feat12 (Mean |SHAP|: 0.7046)
    4. continuous__inspection_feat96 (Mean |SHAP|: 0.6594)
    5. categorical__meta_feat1_27 (Mean |SHAP|: 0.6477)
    6. continuous__inspection_feat94 (Mean |SHAP|: 0.4275)
    7. categorical__meta_feat4_41 (Mean |SHAP|: 0.4106)
    8. continuous__inspection_feat28 (Mean |SHAP|: 0.3262)
    9. categorical__meta_feat1_2 (Mean |SHAP|: 0.3146)
    10. categorical__meta_feat2_2 (Mean |SHAP|: 0.2974)

  -- Native Thresholds for Top Features (Aggregated) --
    * categorical__meta_feat4_7: < 2.0000 (Cover: 48556)
    * continuous__inspection_feat95: < 0.1538 (Cover: 38332), < 0.3077 (Cover: 29674), < 0.1615 (Cover: 15664)
    * continuous__inspection_feat12: < 0.0092 (Cover: 33659), < 0.0175 (Cover: 13869), < 0.0262 (Cover: 6016)
    * continuous__inspection_feat

  Global Top Features (according to Mean Ensemble |SHAP|):
    1. categorical__meta_feat1_0 (Mean |SHAP|: 0.0000)
    2. categorical__meta_feat1_1 (Mean |SHAP|: 0.0000)
    3. categorical__meta_feat1_2 (Mean |SHAP|: 0.0000)
    4. categorical__meta_feat1_7 (Mean |SHAP|: 0.0000)
    5. categorical__meta_feat1_14 (Mean |SHAP|: 0.0000)
    6. categorical__meta_feat1_16 (Mean |SHAP|: 0.0000)
    7. categorical__meta_feat1_17 (Mean |SHAP|: 0.0000)
    8. categorical__meta_feat1_18 (Mean |SHAP|: 0.0000)
    9. categorical__meta_feat1_24 (Mean |SHAP|: 0.0000)
    10. categorical__meta_feat1_33 (Mean |SHAP|: 0.0000)

  -- Native Thresholds for Top Features (Aggregated) --




## 4. Final Summary Table

In [5]:
import IPython.display as display
summary_df = pd.DataFrame(final_summary_rows)
display.display(display.HTML(summary_df.to_html(escape=False, index=False)))

with open("../docs/experiments/0826_dongjin_027_global_ensemble_shap.md", "w", encoding="utf-8") as f:
    f.write("# 0826_dongjin_027_global_ensemble_shap\n\n")
    f.write("## Overview\n")
    f.write("Extracted Global SHAP values over the entire Test Set for each inspection type using the saved `models/0825_peace_005_type_expert_fold_ensemble.pkl` bundle.\n")
    f.write("## Global Top 10 Features by Type\n\n")
    f.write(summary_df.to_markdown(index=False))
    f.write("\n")


Inspection Type,Test Samples,Global Top Features (|SHAP|)
0,19491,1. continuous__inspection_feat41 (0.8916) 2. categorical__meta_feat2_1 (0.7688) 3. continuous__inspection_feat24 (0.5144) 4. categorical__meta_feat4_0 (0.4432) 5. categorical__meta_feat1_10 (0.3970) 6. continuous__inspection_feat25 (0.2829) 7. continuous__inspection_feat23 (0.2733) 8. categorical__meta_feat1_12 (0.1898) 9. continuous__inspection_feat26 (0.1814) 10. continuous__inspection_feat42 (0.1582)
1,12351,1. continuous__inspection_feat48 (1.2720) 2. continuous__inspection_feat24 (0.6964) 3. categorical__meta_feat1_22 (0.5671) 4. categorical__meta_feat4_28 (0.4157) 5. continuous__inspection_feat4 (0.3447) 6. continuous__inspection_feat3 (0.3380) 7. continuous__inspection_feat8 (0.3031) 8. continuous__inspection_feat1 (0.2875) 9. continuous__inspection_feat25 (0.2837) 10. continuous__inspection_feat5 (0.2361)
2,20543,1. categorical__meta_feat1_27 (1.0023) 2. continuous__inspection_feat96 (0.9118) 3. continuous__inspection_feat95 (0.6849) 4. continuous__inspection_feat12 (0.6188) 5. continuous__inspection_feat22 (0.5021) 6. categorical__meta_feat4_41 (0.4317) 7. continuous__inspection_feat1 (0.3846) 8. categorical__meta_feat4_3 (0.3536) 9. continuous__inspection_feat18 (0.3235) 10. categorical__meta_feat1_19 (0.3234)
3,34939,1. categorical__meta_feat4_7 (0.7811) 2. continuous__inspection_feat95 (0.7195) 3. continuous__inspection_feat12 (0.7046) 4. continuous__inspection_feat96 (0.6594) 5. categorical__meta_feat1_27 (0.6477) 6. continuous__inspection_feat94 (0.4275) 7. categorical__meta_feat4_41 (0.4106) 8. continuous__inspection_feat28 (0.3262) 9. categorical__meta_feat1_2 (0.3146) 10. categorical__meta_feat2_2 (0.2974)
4,728,1. categorical__meta_feat1_0 (0.0000) 2. categorical__meta_feat1_1 (0.0000) 3. categorical__meta_feat1_2 (0.0000) 4. categorical__meta_feat1_7 (0.0000) 5. categorical__meta_feat1_14 (0.0000) 6. categorical__meta_feat1_16 (0.0000) 7. categorical__meta_feat1_17 (0.0000) 8. categorical__meta_feat1_18 (0.0000) 9. categorical__meta_feat1_24 (0.0000) 10. categorical__meta_feat1_33 (0.0000)
